# Setup

In [37]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import pickle

PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = PROJECT_ROOT / "db" / "app.db"

print("DB_PATH:", DB_PATH)

DB_PATH: /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/app.db


In [38]:
# load documents and embeddings
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT
        d.id,
        d.clean_text,
        d.publication_year,
        e.embedding
    FROM documents d
    JOIN document_embeddings e
        ON d.id = e.document_id
    WHERE e.model_name = ?
    ORDER BY d.id
""", conn, params=["all-MiniLM-L6-v2"])

conn.close()

print("Loaded rows:", len(df))
display(df.head())

Loaded rows: 738


,id,clean_text,publication_year,embedding
0,1,Study protocol: Comparison of different risk p...,2020,b'\x80\x05\x95u\x06\x00\x00\x00\x00\x00\x00\x8...
1,2,"Mental health, sleep quality and quality of li...",2020,b'\x80\x05\x95u\x06\x00\x00\x00\x00\x00\x00\x8...
2,3,Parent perspectives on food allergy management...,2020,b'\x80\x05\x95u\x06\x00\x00\x00\x00\x00\x00\x8...
3,4,COVID-19 critical illness in Sweden: character...,2020,b'\x80\x05\x95u\x06\x00\x00\x00\x00\x00\x00\x8...
4,5,Geolocated Twitter-based population mobility i...,2020,b'\x80\x05\x95u\x06\x00\x00\x00\x00\x00\x00\x8...


In [39]:
# Reconstruct embedding matrix
df["embedding_vec"] = df["embedding"].apply(pickle.loads)

X = np.vstack(df["embedding_vec"].values)

print("Embedding matrix shape:", X.shape)
print("Dtype:", X.dtype)
print("Years:")
display(df["publication_year"].value_counts().sort_index())

Embedding matrix shape: (738, 384)
Dtype: float32
Years:


publication_year
2020    450
2021    261
2022     27
Name: count, dtype: int64

# K-Means Clusttering

In [40]:
# normalize embeddings
from sklearn.preprocessing import normalize

X_norm = normalize(X, norm="l2")

print("Check norms (should all be ~1):")
print(np.linalg.norm(X_norm[:5], axis=1))

Check norms (should all be ~1):
[0.99999994 1.         1.         1.         1.        ]


In [5]:
# choose k via elbow
from sklearn.cluster import KMeans

ks = list(range(5, 31, 5))  # 5, 10, 15, 20, 25, 30
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_norm)
    inertias.append(km.inertia_)
    print(f"k={k} | inertia={km.inertia_:.2f}")

elbow_df = pd.DataFrame({"k": ks, "inertia": inertias})
display(elbow_df)

k=5 | inertia=460.21
k=10 | inertia=427.32
k=15 | inertia=407.12
k=20 | inertia=394.62
k=25 | inertia=384.27
k=30 | inertia=375.99


,k,inertia
0,5,460.207184
1,10,427.320160
2,15,407.121918
3,20,394.622803
4,25,384.274506
5,30,375.993835


In [41]:
# Run KMC(15)
K = 15

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20
)

df["cluster"] = kmeans.fit_predict(X_norm)

print("Cluster counts:")
display(df["cluster"].value_counts().sort_index())

Cluster counts:


cluster
0     70
1     31
2     47
3     25
4     33
5     56
6     70
7     46
8     68
9     23
10    31
11    63
12    87
13    32
14    56
Name: count, dtype: int64

In [42]:
# inspect clusters
for c in range(K):
    print(f"\n=== Cluster {c} ===")
    sample_titles = df[df["cluster"] == c]["clean_text"].head(5)
    for t in sample_titles:
        print("-", t[:120])


=== Cluster 0 ===
- Public Involvement & Engagement in health inequalities research on COVID-19 pandemic: a case study of CIDACS/FIOCRUZ BAH
- [COVID-19 and its socio-cultural imagery in Latin America: a tool for public health]. OBJECTIVE: Identify the content an
- Occupational health in the framework of the COVID-19 pandemic: a scoping review. OBJECTIVE: To collect the available evi
- [Knowledge of COVID-19 and hand washing]. BACKGROUND: The best way to prevent counting the COVID-19 is hand washing. How
- Health workers as hate crimes targets during COVID-19 outbreak in the Americas. Many health workers in the Americas, esp

=== Cluster 1 ===
- Covid-19 ICU remote-learning course (CIRLC): Rapid ICU remote training for frontline health professionals during the COV
- Virtual Flipped Class and Laboratories for Medical Electronics Course. This paper describes the adaptation of a flipped 
- Online Laboratory Experiment Learning Module for Biomedical Engineering Physiological Laboratory Co

# Topic Extraction (cTFIDF-esque approach)

In [43]:
# Vectorization
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),  # unigrams + bigrams
    min_df=5
)

X_counts = vectorizer.fit_transform(df["clean_text"])

feature_names = np.array(vectorizer.get_feature_names_out())

print("Count matrix shape:", X_counts.shape)

Count matrix shape: (738, 3394)


In [44]:
cluster_term_matrix = np.zeros((K, X_counts.shape[1]))

for c in range(K):
    cluster_mask = (df["cluster"].to_numpy() == c)
    cluster_term_matrix[c] = np.asarray(X_counts[cluster_mask].sum(axis=0)).ravel()

# term frequency within cluster
tf = cluster_term_matrix / cluster_term_matrix.sum(axis=1, keepdims=True)

# inverse document frequency across clusters
doc_freq = (cluster_term_matrix > 0).sum(axis=0)
idf = np.log((K + 1) / (doc_freq + 1)) + 1

# c-TF-IDF
c_tf_idf = tf * idf

In [45]:
# Top words per cluster
TOP_N = 10

cluster_topics = {}

for c in range(K):
    top_idx = np.argsort(c_tf_idf[c])[::-1][:TOP_N]
    top_terms = feature_names[top_idx]
    cluster_topics[c] = top_terms

for c in range(K):
    print(f"\n=== Cluster {c} Top Terms ===")
    print(", ".join(cluster_topics[c]))


=== Cluster 0 Top Terms ===
19, covid, covid 19, health, pandemic, care, women, study, workers, knowledge

=== Cluster 1 Top Terms ===
students, learning, online, student, teaching, covid, 19, covid 19, course, pandemic

=== Cluster 2 Top Terms ===
covid, 19, covid 19, sars, cov, sars cov, patients, infection, cov infection, respiratory

=== Cluster 3 Top Terms ===
images, covid, 19, covid 19, chest, ray, ray images, chest ray, deep, deep learning

=== Cluster 4 Top Terms ===
19, covid, covid 19, stock, returns, markets, growth, pandemic, financial, financial crisis

=== Cluster 5 Top Terms ===
covid, 19, covid 19, anxiety, mental, health, psychological, pandemic, stress, sleep

=== Cluster 6 Top Terms ===
19, covid, covid 19, cases, pandemic, model, health, care, data, colombia

=== Cluster 7 Top Terms ===
covid, 19, covid 19, blockchain, telehealth, health, care, pandemic, services, remote

=== Cluster 8 Top Terms ===
19, covid, covid 19, cov, sars, sars cov, respiratory, patients, 

In [46]:
# Create Labels
topic_labels = {
    0: "Public Health & Workforce Impact",
    1: "Online Education & Training",
    2: "Clinical Virology & Infection",
    3: "Medical Imaging & Deep Learning",
    4: "Financial Markets & Economic Impact",
    5: "Mental Health & Psychological Effects",
    6: "Epidemiological Modeling & Case Data",
    7: "Digital Health, Telemedicine & Blockchain",
    8: "Severe Disease & Immune Response",
    9: "Food Systems & Hospitality Impact",
    10: "Prevention, Masks & Environmental Effects",
    11: "Policy, Governance & Public Response",
    12: "Clinical Care & Procedures",
    13: "Social Media & Misinformation",
    14: "Hospitalization & Patient Outcomes",
}

In [47]:
# Attach labels to DF
df["topic_label"] = df["cluster"].map(topic_labels)

display(df[["id", "cluster", "topic_label"]].head())

,id,cluster,topic_label
0,1,6,Epidemiological Modeling & Case Data
1,2,5,Mental Health & Psychological Effects
2,3,9,Food Systems & Hospitality Impact
3,4,14,Hospitalization & Patient Outcomes
4,5,13,Social Media & Misinformation


In [48]:
# Topic Summary
topic_summary = []

for c in range(K):
    topic_summary.append({
        "topic_id": c,
        "topic_label": topic_labels[c],
        "top_terms": ", ".join(cluster_topics[c]),
        "n_docs": int((df["cluster"] == c).sum())
    })

topic_summary_df = pd.DataFrame(topic_summary).sort_values("topic_id")

display(topic_summary_df)

,topic_id,topic_label,top_terms,n_docs
0,0,Public Health & Workforce Impact,"19, covid, covid 19, health, pandemic, care, w...",70
1,1,Online Education & Training,"students, learning, online, student, teaching,...",31
2,2,Clinical Virology & Infection,"covid, 19, covid 19, sars, cov, sars cov, pati...",47
3,3,Medical Imaging & Deep Learning,"images, covid, 19, covid 19, chest, ray, ray i...",25
4,4,Financial Markets & Economic Impact,"19, covid, covid 19, stock, returns, markets, ...",33
5,5,Mental Health & Psychological Effects,"covid, 19, covid 19, anxiety, mental, health, ...",56
6,6,Epidemiological Modeling & Case Data,"19, covid, covid 19, cases, pandemic, model, h...",70
7,7,"Digital Health, Telemedicine & Blockchain","covid, 19, covid 19, blockchain, telehealth, h...",46
8,8,Severe Disease & Immune Response,"19, covid, covid 19, cov, sars, sars cov, resp...",68
9,9,Food Systems & Hospitality Impact,"covid, 19, covid 19, food, pandemic, hospitali...",23


In [49]:
# Time distribution per topic
topic_time = (
    df.groupby(["publication_year", "cluster"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

display(topic_time)

cluster,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
publication_year,,,,,,,,,,,,,,,
2020,55,13,22,10,5,38,56,32,36,10,17,46,53,28,29
2021,13,16,23,14,27,15,14,14,30,12,13,15,30,4,21
2022,2,2,2,1,1,3,0,0,2,1,1,2,4,0,6


In [19]:
conn = sqlite3.connect(DB_PATH)

conn.executescript("""
CREATE TABLE IF NOT EXISTS topics (
    id INTEGER PRIMARY KEY,
    topic_id INTEGER,
    topic_label TEXT,
    top_terms TEXT,
    n_docs INTEGER,
    model_name TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(topic_id, model_name)
);

CREATE TABLE IF NOT EXISTS document_topics (
    id INTEGER PRIMARY KEY,
    document_id INTEGER,
    topic_id INTEGER,
    model_name TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (document_id) REFERENCES documents(id),
    UNIQUE(document_id, topic_id, model_name)
);
""")

conn.commit()

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""", conn)

display(tables)

conn.close()

,name
0,document_embeddings
1,document_topics
2,documents
3,topics


In [20]:
# create new topic tables
conn = sqlite3.connect(DB_PATH)

with open(PROJECT_ROOT / "db" / "schema.sql", "r") as f:
    schema_sql = f.read()

conn.executescript(schema_sql)
conn.commit()
conn.close()

print("Schema updated.")

Schema updated.


In [21]:
# Prepare topic records
TOPIC_MODEL_NAME = "kmeans_all-MiniLM-L6-v2_k15"

topics_to_insert = topic_summary_df.copy()
topics_to_insert["model_name"] = TOPIC_MODEL_NAME

display(topics_to_insert.head())

,topic_id,topic_label,top_terms,n_docs,model_name
0,0,Public Health & Workforce Impact,"19, covid, covid 19, health, pandemic, care, w...",70,kmeans_all-MiniLM-L6-v2_k15
1,1,Online Education & Training,"students, learning, online, student, teaching,...",31,kmeans_all-MiniLM-L6-v2_k15
2,2,Clinical Virology & Infection,"covid, 19, covid 19, sars, cov, sars cov, pati...",47,kmeans_all-MiniLM-L6-v2_k15
3,3,Medical Imaging & Deep Learning,"images, covid, 19, covid 19, chest, ray, ray i...",25,kmeans_all-MiniLM-L6-v2_k15
4,4,Financial Markets & Economic Impact,"19, covid, covid 19, stock, returns, markets, ...",33,kmeans_all-MiniLM-L6-v2_k15


In [50]:
# Clear old topic rows for this model (important for reruns)
conn = sqlite3.connect(DB_PATH)

conn.execute(
    "DELETE FROM document_topics WHERE model_name = ?",
    (TOPIC_MODEL_NAME,)
)
conn.execute(
    "DELETE FROM topics WHERE model_name = ?",
    (TOPIC_MODEL_NAME,)
)

conn.commit()
conn.close()

print("Cleared old topic rows for model:", TOPIC_MODEL_NAME)

Cleared old topic rows for model: kmeans_all-MiniLM-L6-v2_k15


In [51]:
# Insert topics into SQLite
conn = sqlite3.connect(DB_PATH)

insert_topics_sql = """
INSERT OR IGNORE INTO topics (
    topic_id,
    topic_label,
    top_terms,
    n_docs,
    model_name
)
VALUES (?, ?, ?, ?, ?)
"""

topic_rows = list(
    topics_to_insert[["topic_id", "topic_label", "top_terms", "n_docs", "model_name"]]
    .itertuples(index=False, name=None)
)

before_n = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM topics WHERE model_name = ?",
    conn,
    params=[TOPIC_MODEL_NAME]
)["n"].iloc[0]

conn.executemany(insert_topics_sql, topic_rows)
conn.commit()

after_n = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM topics WHERE model_name = ?",
    conn,
    params=[TOPIC_MODEL_NAME]
)["n"].iloc[0]

print("Topics before insert:", before_n)
print("Topics attempted:", len(topic_rows))
print("Topics after insert:", after_n)

conn.close()

Topics before insert: 0
Topics attempted: 15
Topics after insert: 15


In [52]:
# Rebuild document-topic assignments from the current clustered dataframe
doc_topics_df = df[["id", "cluster"]].copy()
doc_topics_df = doc_topics_df.rename(columns={"id": "document_id", "cluster": "topic_id"})

doc_topics_df["document_id"] = doc_topics_df["document_id"].astype(int)
doc_topics_df["topic_id"] = doc_topics_df["topic_id"].astype(int)
doc_topics_df["model_name"] = TOPIC_MODEL_NAME

display(doc_topics_df.head())
print("Document-topic rows:", len(doc_topics_df))

,document_id,topic_id,model_name
0,1,6,kmeans_all-MiniLM-L6-v2_k15
1,2,5,kmeans_all-MiniLM-L6-v2_k15
2,3,9,kmeans_all-MiniLM-L6-v2_k15
3,4,14,kmeans_all-MiniLM-L6-v2_k15
4,5,13,kmeans_all-MiniLM-L6-v2_k15


Document-topic rows: 738


In [53]:
# Delete broken rows for this model, then reinsert cleanly
conn = sqlite3.connect(DB_PATH)

conn.execute(
    "DELETE FROM document_topics WHERE model_name = ?",
    [TOPIC_MODEL_NAME]
)
conn.commit()

insert_doc_topics_sql = """
INSERT OR IGNORE INTO document_topics (
    document_id,
    topic_id,
    model_name
)
VALUES (?, ?, ?)
"""

doc_topic_rows = list(
    doc_topics_df[["document_id", "topic_id", "model_name"]]
    .itertuples(index=False, name=None)
)

conn.executemany(insert_doc_topics_sql, doc_topic_rows)
conn.commit()

n_doc_topics = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM document_topics WHERE model_name = ?",
    conn,
    params=[TOPIC_MODEL_NAME]
)["n"].iloc[0]

print("Reinserted document_topics rows:", n_doc_topics)

conn.close()

Reinserted document_topics rows: 738


In [54]:
# Validate topic storage and joins
conn = sqlite3.connect(DB_PATH)

sample_join = pd.read_sql_query("""
    SELECT
        d.id AS document_id,
        dt.topic_id,
        t.topic_label,
        d.publication_year,
        substr(d.title, 1, 120) AS title_preview
    FROM document_topics dt
    JOIN documents d
        ON dt.document_id = d.id
    JOIN topics t
        ON dt.topic_id = t.topic_id
       AND dt.model_name = t.model_name
    WHERE dt.model_name = ?
    ORDER BY d.id
    LIMIT 10
""", conn, params=[TOPIC_MODEL_NAME])

orphan_check = pd.read_sql_query("""
    SELECT COUNT(*) AS n_orphans
    FROM document_topics dt
    LEFT JOIN documents d
        ON dt.document_id = d.id
    WHERE dt.model_name = ?
      AND d.id IS NULL
""", conn, params=[TOPIC_MODEL_NAME])

conn.close()

print("Orphan rows:")
display(orphan_check)

print("\nSample joined records:")
display(sample_join)

Orphan rows:


,n_orphans
0,0



Sample joined records:


,document_id,topic_id,topic_label,publication_year,title_preview
0,1,6,Epidemiological Modeling & Case Data,2020,Study protocol: Comparison of different risk p...
1,2,5,Mental Health & Psychological Effects,2020,"Mental health, sleep quality and quality of li..."
2,3,9,Food Systems & Hospitality Impact,2020,Parent perspectives on food allergy management...
3,4,14,Hospitalization & Patient Outcomes,2020,COVID-19 critical illness in Sweden: character...
4,5,13,Social Media & Misinformation,2020,Geolocated Twitter-based population mobility i...
5,6,0,Public Health & Workforce Impact,2020,Public Involvement & Engagement in health ineq...
6,7,6,Epidemiological Modeling & Case Data,2020,Using data linkage to monitor COVID-19 vaccina...
7,8,6,Epidemiological Modeling & Case Data,2020,A living mapping review for COVID-19 funded re...
8,9,0,Public Health & Workforce Impact,2020,[COVID-19 and its socio-cultural imagery in La...
9,10,10,"Prevention, Masks & Environmental Effects",2020,"[Risks, contamination and prevention against C..."


In [55]:
conn = sqlite3.connect(DB_PATH)

doc_topics_type_check = pd.read_sql_query("""
    SELECT
        document_id,
        typeof(document_id) AS document_id_type,
        topic_id,
        typeof(topic_id) AS topic_id_type,
        model_name
    FROM document_topics
    WHERE model_name = ?
    LIMIT 10
""", conn, params=[TOPIC_MODEL_NAME])

documents_type_check = pd.read_sql_query("""
    SELECT
        id,
        typeof(id) AS id_type
    FROM documents
    LIMIT 10
""", conn)

conn.close()

print("document_topics types:")
display(doc_topics_type_check)

print("\ndocuments types:")
display(documents_type_check)

document_topics types:


,document_id,document_id_type,topic_id,topic_id_type,model_name
0,1,integer,6,integer,kmeans_all-MiniLM-L6-v2_k15
1,2,integer,5,integer,kmeans_all-MiniLM-L6-v2_k15
2,3,integer,9,integer,kmeans_all-MiniLM-L6-v2_k15
3,4,integer,14,integer,kmeans_all-MiniLM-L6-v2_k15
4,5,integer,13,integer,kmeans_all-MiniLM-L6-v2_k15
5,6,integer,0,integer,kmeans_all-MiniLM-L6-v2_k15
6,7,integer,6,integer,kmeans_all-MiniLM-L6-v2_k15
7,8,integer,6,integer,kmeans_all-MiniLM-L6-v2_k15
8,9,integer,0,integer,kmeans_all-MiniLM-L6-v2_k15
9,10,integer,10,integer,kmeans_all-MiniLM-L6-v2_k15



documents types:


,id,id_type
0,8,integer
1,469,integer
2,430,integer
3,530,integer
4,738,integer
5,737,integer
6,736,integer
7,735,integer
8,734,integer
9,733,integer


In [ ]:
# 	TODO: consider storing clustering hyperparameters / vectorizer settings
